In [ ]:
def montrer(uid, subset, seulement_rates=True, **params):
    p = next((LUNA / subset).rglob(f"{uid}.mhd"))
    im = resample_isotropic(sitk.ReadImage(str(p)), cfg["resample_mm"])
    v = sitk.GetArrayFromImage(im)
    m = segment_lungs(v, **params)
    for z, y, x in nodule_voxels(im, ann[ann.seriesuid == uid]):
        dedans = nodules_inside(m, [(z, y, x)]) == 1
        if seulement_rates and dedans:
            continue
        plt.figure(figsize=(6, 6))
        plt.imshow(v[z], cmap="gray", vmin=-1000, vmax=400)
        plt.contour(m[z].astype(float), levels=[0.5], colors="r")
        plt.plot(x, y, "c+", ms=16)
        plt.xlim(x - 60, x + 60); plt.ylim(y + 60, y - 60)
        plt.title(f"{uid[-8:]}  z={z}  nodule dans le masque : {dedans}")
        plt.axis("off")

ratees = res[res.dans_masque < res.nodules]
print(ratees if len(ratees) else "Aucun nodule perdu")
for _, r in ratees.iterrows():
    montrer(r.seriesuid, r.subset, **params)

In [ ]:
res = []
for _, r in patients[patients.split == "dev"].iterrows():
    p = next((LUNA / r.subset).rglob(f"{r.seriesuid}.mhd"))
    im = resample_isotropic(sitk.ReadImage(str(p)), cfg["resample_mm"])
    m = segment_lungs(sitk.GetArrayFromImage(im), **params)
    v = nodule_voxels(im, ann[ann.seriesuid == r.seriesuid])
    _, n_comp = ndi.label(m)
    res.append({"seriesuid": r.seriesuid, "subset": r.subset,
                "volume_L": round(m.sum() / 1e6, 2), "composantes": n_comp,
                "nodules": len(v), "dans_masque": nodules_inside(m, v)})
res = pd.DataFrame(res)
res.to_csv(ROOT / "results" / "segmentation_eval.csv", index=False)
print(res)
print("Nodules retrouvés :", res.dans_masque.sum(), "/", res.nodules.sum())

In [ ]:
tests = []
for thr in [-400, -320]:
    for rc in [4, 8, 12]:
        t0 = time.time()
        m = segment_lungs(vol, thr=thr, r_close=rc)
        tests.append({"seuil": thr, "r_close": rc, "volume_L": round(m.sum()/1e6, 2),
                      "nodules_dans_masque": nodules_inside(m, vox),
                      "temps_s": round(time.time() - t0)})
df_tests = pd.DataFrame(tests)
df_tests.to_csv(ROOT / "results" / "tests_parametres.csv", index=False)
df_tests

In [ ]:
from skimage.filters import threshold_otsu
t_otsu = threshold_otsu(vol[vol > -1500][::50])
print("Seuil d'Otsu :", round(float(t_otsu)), "HU")

for thr in [t_otsu, -320]:
    m = segment_lungs(vol, thr=thr)
    print(f"seuil {thr:.0f} : {m.sum()/1e6:.2f} L, "
          f"{nodules_inside(m, vox)} nodules sur {len(vox)}")

In [ ]:
from scipy import ndimage as ndi
k = int(np.median(np.where(mask.any(axis=(1, 2)))[0]))     # coupe au milieu du thorax
orig = vol[k].astype(np.float32)
med = ndi.median_filter(orig, size=3)
gau = ndi.gaussian_filter(orig, sigma=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, t in zip(axes, [orig, med, gau], ["Original", "Médian 3x3", "Gaussien σ=1"]):
    ax.imshow(im, cmap="gray", vmin=-1000, vmax=400)
    ax.set_title(t); ax.axis("off")
plt.show()

for nom, im in [("original", orig), ("médian", med), ("gaussien", gau)]:
    print(f"écart-type dans les poumons ({nom}) : {im[mask[k]].std():.1f} HU")

In [ ]:
nod = ann[ann.seriesuid == row.seriesuid]
vox = nodule_voxels(img_rs, nod)
print(f"{nodules_inside(mask, vox)} nodules sur {len(vox)} dans le masque")

In [ ]:
zs = np.where(mask.any(axis=(1, 2)))[0]
idx = np.linspace(zs.min() + 10, zs.max() - 10, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, k in zip(axes.ravel(), idx):
    ax.imshow(vol[k], cmap="gray", vmin=-1000, vmax=400)
    ax.contour(mask[k].astype(float), levels=[0.5], colors="r", linewidths=1)
    ax.set_title(f"z = {k}"); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
t0 = time.time()
mask = segment_lungs(vol, **params)
print(f"{time.time() - t0:.0f} s, volume pulmonaire : {mask.sum() / 1e6:.2f} L")

In [ ]:
row = patients[patients.split == "dev"].iloc[0]
path = next((LUNA / row.subset).rglob(f"{row.seriesuid}.mhd"))
img = sitk.ReadImage(str(path))
img_rs = resample_isotropic(img, cfg["resample_mm"])
vol = sitk.GetArrayFromImage(img_rs)
print("avant :", img.GetSize(), img.GetSpacing())
print("après :", img_rs.GetSize(), img_rs.GetSpacing())

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, time, yaml
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage as ndi

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.segmentation import resample_isotropic, segment_lungs, nodule_voxels, nodules_inside

LUNA = ROOT / "data" / "luna16"
patients = pd.read_csv(ROOT / "results" / "patients.csv")
ann = pd.read_csv(LUNA / "annotations.csv")

cfg = yaml.safe_load(open(ROOT / "config.yaml"))
params = dict(thr=cfg["thr"], r_open=cfg["r_open"], r_close=cfg["r_close"])
print(cfg)